# 😊 PHILIA — Facial Emotion Fine-Tuning
**Base model:** `mo-thecreator/vit-Facial-Expression-Recognition` (pretrained on facial expression data) → fine-tuned on FER2013  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Loss:** Focal Loss + class weights (handles FER2013 class imbalance)  

## Key features
- **Pretrained facial emotion encoder** — starts from a ViT already trained on facial expression recognition, so the attention maps already focus on emotion-relevant face regions (eyes, mouth corners)
- **FER2013 fine-tuning** — adapts to the full 7-class FER2013 dataset
- **Data Augmentation** — random flips, rotation ±15°, brightness/contrast jitter
- **Focal Loss + Class Weights** — FER2013 is imbalanced (happy >> disgust/fear)
- **Early Stopping** — patience=3 epochs to prevent overfitting

> **Windows:** `dataloader_num_workers=0` required — Windows spawn multiprocessing cannot pickle custom Trainer subclasses.


In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate scikit-learn Pillow

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/facial_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Load FER2013 from HuggingFace ─────────────────────────────────────
from datasets import load_dataset
import collections

# FER2013: 7 emotions, pre-cropped 48x48 face images
# 28,709 train | 3,589 val | 3,589 test
print('Loading FER2013...')
fer = load_dataset('3una/Fer2013')
print('Splits:', fer)

# Get label names from dataset feature
label_feature = fer['train'].features.get('label')
label_names_raw = label_feature.names if hasattr(label_feature, 'names') else \
    ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

print('Dataset label names:', label_names_raw)
print('Train dist:', dict(collections.Counter(fer['train']['label'])))

In [ ]:
# ── Cell 5: Map FER2013 labels to canonical + set up label dicts ──────────────
# FER2013 label mapping (some datasets use 'angry', others 'anger')
CANONICAL = {
    'angry':    'angry',   'anger':   'angry',
    'disgust':  'disgust',
    'fear':     'fear',    'fearful': 'fear',
    'happy':    'happy',   'happiness':'happy',
    'sad':      'sad',     'sadness': 'sad',
    'surprise': 'surprise','surprised':'surprise',
    'neutral':  'neutral',
}
LABELS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Map integer label_id → raw name → canonical → final int
def remap_label(example):
    raw = label_names_raw[example['label']].lower()
    canonical = CANONICAL.get(raw, None)
    if canonical is None:
        return {'new_label': -1}   # will be filtered out
    return {'new_label': LABEL2ID[canonical]}

for split in fer:
    fer[split] = fer[split].map(remap_label)
    fer[split] = fer[split].filter(lambda x: x['new_label'] >= 0)

print('After remapping:')
for split in fer:
    cnt = dict(collections.Counter([LABELS[l] for l in fer[split]['new_label']]))
    print(f'  {split}: {len(fer[split])} samples | {cnt}')

In [ ]:
# ── Cell 6: Load ViT feature extractor and preprocess images ──────────────────
from transformers import ViTImageProcessor
from PIL import Image
import numpy as np

MODEL_CHECKPOINT = 'mo-thecreator/vit-Facial-Expression-Recognition'
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def preprocess(batch):
    # Resize to 224x224 RGB (ViT requirement)
    images = []
    for img in batch['image']:
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img = img.convert('RGB').resize((224, 224))
        images.append(img)
    inputs = processor(images=images, return_tensors='np')
    inputs['labels'] = batch['new_label']
    return inputs

print('Preprocessing images...')
train_ds = fer['train'].map(preprocess, batched=True, batch_size=64,
                            remove_columns=['image', 'label', 'new_label'])
val_ds   = fer['validation'].map(preprocess, batched=True, batch_size=64,
                                 remove_columns=['image', 'label', 'new_label']) \
            if 'validation' in fer else \
           fer['test'].map(preprocess, batched=True, batch_size=64,
                           remove_columns=['image', 'label', 'new_label'])
test_ds  = fer['test'].map(preprocess, batched=True, batch_size=64,
                           remove_columns=['image', 'label', 'new_label'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Done. Train:', len(train_ds), '| Val:', len(val_ds), '| Test:', len(test_ds))

In [ ]:
# ── Cell 7: Load ViT model ─────────────────────────────────────────────────────
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

total   = sum(p.numel() for p in model.parameters())
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total/1e6:.1f}M | Trainable: {trained/1e6:.1f}M')

In [ ]:
# ── Cell 8: Training ──────────────────────────────────────────────────────────
import evaluate, numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)

training_args = TrainingArguments(
    output_dir='/content/vit_facial_emotion',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
    fp16=True,
    dataloader_num_workers=0,  # Windows: avoid spawn multiprocessing pickling crash
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── Cell 9: Evaluate on test set ─────────────────────────────────────────────
results = trainer.evaluate(test_ds)
print('Test results:', results)

from sklearn.metrics import classification_report
pred_output = trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
print(classification_report(
    pred_output.label_ids, preds,
    target_names=LABELS, digits=3
))

In [ ]:
# ── Cell 10: Save to Google Drive ─────────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 11: (Optional) Push to HuggingFace Hub ───────────────────────────────
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-facial-emotion')
# processor.push_to_hub('YOUR_HF_USERNAME/philia-facial-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')